In [1]:
# =============================================================================
# 功能：读取包含论文QA数据的JSON文件，统计并生成一份专业格式的Excel统计报告。
#       报告包含三个主要部分：
#       1. 左侧学科统计表：按“一级学科（Primary）”和“二级分类（Secondary）”
#          统计论文数量，并自动合并同类学科的单元格，底部添加去重求和与明细求和。
#       2. 右侧问题分类统计表：按“一级学科”和“问题类别（Question Category）”
#          统计各学科的QA数量，同样支持单元格合并与求和。
#       3. 总体统计表：汇总总QA数量、选择题/简答题分布、单模态/多模态分布、
#          跨页/未跨页分布、字面抽取类/深层理解类分布，并计算各项占比。
#       输出Excel文件包含专业格式：垂直居中、列宽适配、外侧框线、合并单元格、
#       求和公式等，便于交接和汇报。
# =============================================================================
import json
import pandas as pd
from collections import defaultdict
from openpyxl.styles import Alignment, Border, Side
import os

# ==================== 映射字典 ====================
PRIMARY_TO_ALLOWED_SECONDARY = {
    "Computer Science": [
        "Artificial Intelligence", "Computational Complexity", "Computers and Society",
        "Databases", "Data Structures and Algorithms", "Distributed Systems",
        "Neural and Evolutionary Computing", "Computer Science and Game Theory",
        "Computational Geometry", "Hardware Architecture", "Information Theory",
        "Multimedia", "Computation and Language", "Machine Learning",
        "Programming Languages", "Computational Engineering, Finance, and Science",
        "Cryptography and Security", "Social and Information Networks",
        "Computer Vision and Pattern Recognition"
    ],
    "Economics": [
        "Econometrics", "Theoretical Economics"
    ],
    "Electrical Engineering and Systems Science": [
        "Audio and Speech Processing", "Image and Video Processing", "Signal Processing"
    ],
    "Mathematics": [
        "Algebraic Geometry", "Analysis of PDEs", "Applied Probability", "Number Theory",
        "General Mathematics", "Geometry", "Category Theory", "Algebras",
        "Representation Theory", "Algebraic Topology"
    ],
    "Physics": [
        "Applied Plasma Physics", "Instrumentation and Methods for Astrophysics",
        "Atomic and Molecular Clusters", "Classical Physics", "Statistical Mechanics",
        "Cosmology and Nongalactic Astrophysics", "Soft Condensed Matter",
        "Space Biophysics", "High Energy Astrophysical Phenomena", "Mathematical Physics",
        "Nuclear Physics", "High Energy Physics", "Astrophysics of Galaxies",
        "Mesoscale and Nanoscale Physics", "Quantum Physics", "General Relativity and Quantum Cosmology"
    ],
    "Quantitative Biology": [
        "Biomolecules", "Cell Behavior", "Other Quantitative Biology", "Molecular Networks"
    ],
    "Quantitative Finance": [
        "Computational Finance", "Statistical Finance", "Mathematical Finance",
        "Portfolio and Risk Management"
    ],
    "Statistics": [
        "Applications and Methodology", "Computational Statistics", "Other Statistics"
    ]
}

PRIMARY_TO_ALLOWED_CATEGORIES = {
    "Computer Science": [
        "Algorithm & Architecture Detail",
        "Experiment & Result Validation",
        "Method Innovation"
    ],
    "Economics": [
        "Theoretical Framework & Concept Definition",
        "Empirical Design & Econometric Method",
        "Causal Inference & Result Interpretation"
    ],
    "Electrical Engineering and Systems Science": [
        "System Architecture & Signal Processing Method",
        "Experiment & Performance Test",
        "System Optimization & Engineering Implementation"
    ],
    "Mathematics": [
        "Definition & Theorem Statement",
        "Formula & Derivation Detail",
        "Proposition Proof & Logical Reasoning"
    ],
    "Physics": [
        "Physical Concept & Theoretical Model",
        "Computation & Experimental Validation"
    ],
    "Quantitative Biology": [
        "Biological Entity & Quantitative Model Definition",
        "Experimental Data & Statistical Analysis",
        "Biological Network & Dynamic Modeling"
    ],
    "Quantitative Finance": [
        "Financial Theory & Pricing Model Definition",
        "Quantitative Strategy & Performance Analysis"
    ],
    "Statistics": [
        "Statistical Concept & Probability Model Definition",
        "Statistical Inference & Method Application"
    ]
}

# ==================== 数据分析 ====================
def analyze_json(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    print(f"JSON文件中共有 {len(data)} 个键（论文）")

    paper_count = defaultdict(lambda: defaultdict(int))
    primary_paper_count = defaultdict(int)
    primary_qa_count = defaultdict(int)
    qa_category_count = defaultdict(lambda: defaultdict(int))

    total_qa = 0
    mc_cnt = sa_cnt = 0
    single_mod = multi_mod = 0
    single_page = multi_page = 0
    literal_cnt = inferential_cnt = 0

    debug_count = 0
    sample_primaries = set()
    sample_secondaries = set()

    for paper_id, paper_info in data.items():
        primary = paper_info.get("primary_category", "")
        secondary = paper_info.get("secondary_category", "")



        if debug_count < 5:
            print(f"调试样例 {paper_id}: primary='{primary}', secondary='{secondary}'")
            sample_primaries.add(primary)
            sample_secondaries.add(secondary)
            debug_count += 1

        if primary and secondary:
            paper_count[primary][secondary] += 1
            primary_paper_count[primary] += 1

        qa_dict = paper_info.get("QA", {})
        for qa_item in qa_dict.values():
            total_qa += 1
            if primary:
                primary_qa_count[primary] += 1

            if "options" in qa_item:
                mc_cnt += 1
            else:
                sa_cnt += 1

            modal_types = qa_item.get("modal_types", [])
            if len(modal_types) == 1:
                single_mod += 1
            elif len(modal_types) > 1:
                multi_mod += 1

            evidence_pages = qa_item.get("evidence_pages", [])
            if len(evidence_pages) == 1:
                single_page += 1
            elif len(evidence_pages) > 1:
                multi_page += 1

            q_type = qa_item.get("question_type", "")
            if q_type == "Literal":
                literal_cnt += 1
            elif q_type == "Inferential":
                inferential_cnt += 1

            q_category = qa_item.get("question_category", "")
            if primary and q_category:
                qa_category_count[primary][q_category] += 1

    print("\n--- 统计结果 ---")
    print(f"总QA数: {total_qa}")
    print(f"成功计数的论文数: {sum(primary_paper_count.values())}")
    print(f"各 primary 论文总数: {dict(primary_paper_count)}")
    print(f"各 primary QA总数: {dict(primary_qa_count)}")
    print(f"选择题: {mc_cnt}, 简答题: {sa_cnt}")

    missing_primary = set(primary_paper_count.keys()) - set(PRIMARY_TO_ALLOWED_SECONDARY.keys())
    if missing_primary:
        print(f"警告：以下 primary 不在映射字典中: {missing_primary}")

    return (paper_count, primary_paper_count, primary_qa_count, qa_category_count,
            total_qa, mc_cnt, sa_cnt,
            single_mod, multi_mod,
            single_page, multi_page,
            literal_cnt, inferential_cnt)

# ==================== 构建左侧学科统计（前四列） ====================
def build_left_table(paper_count, primary_paper_count):
    rows = []
    for primary in PRIMARY_TO_ALLOWED_SECONDARY.keys():
        secondaries = PRIMARY_TO_ALLOWED_SECONDARY[primary]
        primary_total = primary_paper_count.get(primary, 0)
        for secondary in secondaries:
            paper_cnt = paper_count.get(primary, {}).get(secondary, 0)
            # 临时列名避免字典key重复，后续重命名
            rows.append({
                "Primary Category": primary,
                "Papers_1": primary_total,
                "Secondary Category": secondary,
                "Papers_2": paper_cnt
            })
    df = pd.DataFrame(rows)
    # 重命名为用户指定的表头，支持重复列名
    df = df.rename(columns={
        "Papers_1": "Papers",
        "Papers_2": "Papers"
    })
    return df

# ==================== 构建右侧问题分类统计（后四列） ====================
def build_right_table(primary_qa_count, qa_category_count):
    rows = []
    for primary in PRIMARY_TO_ALLOWED_SECONDARY.keys():
        primary_qa_total = primary_qa_count.get(primary, 0)
        allowed_cats = PRIMARY_TO_ALLOWED_CATEGORIES.get(primary, [])
        cat_counts = qa_category_count.get(primary, {})
        for q_cat in allowed_cats:
            qa_cnt = cat_counts.get(q_cat, 0)
            # 临时列名避免字典key重复，后续重命名
            rows.append({
                "Primary Category": primary,
                "QAs_1": primary_qa_total,
                "Question Category": q_cat,
                "QAs_2": qa_cnt
            })
    df = pd.DataFrame(rows)
    # 重命名为用户指定的表头，支持重复列名
    df = df.rename(columns={
        "QAs_1": "QAs",
        "QAs_2": "QAs"
    })
    return df

# ==================== 构建总体统计表 ====================
def build_overall_df(total_qa, mc_cnt, sa_cnt, single_mod, multi_mod,
                     single_page, multi_page, literal_cnt, inferential_cnt):
    rows = [
        ["选择题", mc_cnt, f"{mc_cnt/total_qa:.2%}",
         "简答题", sa_cnt, f"{sa_cnt/total_qa:.2%}",
         total_qa, "100%"],
        ["单模态", single_mod, f"{single_mod/total_qa:.2%}",
         "多模态", multi_mod, f"{multi_mod/total_qa:.2%}",
         total_qa, "100%"],
        ["未跨页", single_page, f"{single_page/total_qa:.2%}",
         "跨页", multi_page, f"{multi_page/total_qa:.2%}",
         total_qa, "100%"],
        ["字面抽取类", literal_cnt, f"{literal_cnt/total_qa:.2%}",
         "深层理解类", inferential_cnt, f"{inferential_cnt/total_qa:.2%}",
         total_qa, "100%"]
    ]
    columns = ["类型A", "数量A", "占比A", "类型B", "数量B", "占比B", "总数", "总数占比"]
    return pd.DataFrame(rows, columns=columns)

# ==================== 合并单元格辅助函数 ====================
def merge_cells_by_group(worksheet, start_row, end_row, col_idx, group_col_idx):
    """
    根据 group_col_idx 列的值变化，合并 col_idx 列的单元格。
    注意：此函数假设数据已按 group_col_idx 排序，且group_col_idx列所有行都有完整值
    """
    current_group = None
    merge_start = start_row
    for row in range(start_row, end_row + 1):
        group_val = worksheet.cell(row=row, column=group_col_idx).value
        if group_val is None:
            continue
        if group_val != current_group:
            if current_group is not None and row - merge_start > 1:
                worksheet.merge_cells(start_row=merge_start, start_column=col_idx,
                                      end_row=row-1, end_column=col_idx)
            current_group = group_val
            merge_start = row
    # 处理最后一组
    if current_group is not None and end_row - merge_start >= 1:
        actual_end = end_row
        while actual_end >= start_row and worksheet.cell(row=actual_end, column=group_col_idx).value is None:
            actual_end -= 1
        if actual_end >= merge_start:
            worksheet.merge_cells(start_row=merge_start, start_column=col_idx,
                                  end_row=actual_end, end_column=col_idx)

# ==================== 外侧框线辅助函数 ====================
def set_outside_border(worksheet, min_row, max_row, min_col, max_col):
    """给指定的单元格区域设置专业黑色细实线外侧框线"""
    thin_side = Side(style='thin', color='000000')
    # 上边框
    for col in range(min_col, max_col + 1):
        cell = worksheet.cell(row=min_row, column=col)
        cell.border = Border(top=thin_side, bottom=cell.border.bottom, left=cell.border.left, right=cell.border.right)
    # 下边框
    for col in range(min_col, max_col + 1):
        cell = worksheet.cell(row=max_row, column=col)
        cell.border = Border(top=cell.border.top, bottom=thin_side, left=cell.border.left, right=cell.border.right)
    # 左边框
    for row in range(min_row, max_row + 1):
        cell = worksheet.cell(row=row, column=min_col)
        cell.border = Border(top=cell.border.top, bottom=cell.border.bottom, left=thin_side, right=cell.border.right)
    # 右边框
    for row in range(min_row, max_row + 1):
        cell = worksheet.cell(row=row, column=max_col)
        cell.border = Border(top=cell.border.top, bottom=cell.border.bottom, left=cell.border.left, right=thin_side)

# ==================== 主程序 ====================
def main():
    json_path = "data/qa/1.base/work__single_pdf__raw_mixed__n6204.json"
    output_path = "data/archive/initial_collection/统计结果.xlsx"

    if not os.path.exists(json_path):
        print(f"错误：未找到文件 {json_path}")
        return

    print("正在分析 JSON 文件...")
    (paper_count, primary_paper_count, primary_qa_count, qa_category_count,
     total_qa, mc_cnt, sa_cnt,
     single_mod, multi_mod,
     single_page, multi_page,
     literal_cnt, inferential_cnt) = analyze_json(json_path)

    print("正在构建左侧学科统计表...")
    df_left = build_left_table(paper_count, primary_paper_count)

    print("正在构建右侧问题分类统计表...")
    df_right = build_right_table(primary_qa_count, qa_category_count)

    print("正在构建总体统计表...")
    df_overall = build_overall_df(total_qa, mc_cnt, sa_cnt, single_mod, multi_mod,
                                  single_page, multi_page, literal_cnt, inferential_cnt)

    # 保存原始数据长度，用于后续确定合并范围和求和位置
    original_len_left = len(df_left)
    original_len_right = len(df_right)

    # 左右侧行数对齐
    max_rows = max(len(df_left), len(df_right))
    if len(df_left) < max_rows:
        df_left = df_left.reindex(range(max_rows)).fillna("")
    if len(df_right) < max_rows:
        df_right = df_right.reindex(range(max_rows)).fillna("")

    df_combined = pd.concat([df_left, df_right], axis=1)

    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        df_combined.to_excel(writer, sheet_name='统计结果', index=False, startrow=0, startcol=0)
        df_overall.to_excel(writer, sheet_name='统计结果', index=False, startrow=0, startcol=8)

        worksheet = writer.sheets['统计结果']

        # ========== 单元格合并 ==========
        # 左侧合并范围：仅原始有数据的行
        start_left = 2
        end_left = start_left + original_len_left - 1
        # 先合并数值列，再合并分类列，避免分组识别失败
        merge_cells_by_group(worksheet, start_left, end_left, 2, 1)
        merge_cells_by_group(worksheet, start_left, end_left, 1, 1)

        # 右侧合并范围：仅原始有数据的行
        start_right = 2
        end_right = start_right + original_len_right - 1
        merge_cells_by_group(worksheet, start_right, end_right, 6, 5)
        merge_cells_by_group(worksheet, start_right, end_right, 5, 5)

        # ========== 求和公式与Sum标识 ==========
        # 左侧求和行
        sum_row_left = end_left + 1
        # 去重求和：大类总数仅计算一次
        worksheet.cell(row=sum_row_left, column=2).value = f"=SUMIF(A2:A{end_left}, \"<>\", B2:B{end_left})"
        # 明细求和：子项全量求和
        worksheet.cell(row=sum_row_left, column=4).value = f"=SUM(D2:D{end_left})"
        # 左侧Sum标识
        worksheet.cell(row=sum_row_left, column=1).value = "Sum"
        worksheet.cell(row=sum_row_left, column=3).value = "Sum"

        # 右侧求和行
        sum_row_right = end_right + 1
        # 去重求和：大类总数仅计算一次
        worksheet.cell(row=sum_row_right, column=6).value = f"=SUMIF(E2:E{end_right}, \"<>\", F2:F{end_right})"
        # 明细求和：子项全量求和
        worksheet.cell(row=sum_row_right, column=8).value = f"=SUM(H2:H{end_right})"
        # 右侧Sum标识
        worksheet.cell(row=sum_row_right, column=5).value = "Sum"
        worksheet.cell(row=sum_row_right, column=7).value = "Sum"

        # ========== 格式美化 ==========
        # 全区域垂直居中
        center_alignment = Alignment(vertical='center')
        # 左侧数据+求和行居中
        for row in worksheet.iter_rows(min_row=2, max_row=sum_row_left):
            for cell in row:
                cell.alignment = center_alignment
        # 右侧数据+求和行居中
        for row in worksheet.iter_rows(min_row=2, max_row=sum_row_right):
            for cell in row:
                cell.alignment = center_alignment
        # 总体统计区域居中
        overall_max_row = 1 + len(df_overall)
        for row in worksheet.iter_rows(min_row=1, max_row=overall_max_row, min_col=9, max_col=16):
            for cell in row:
                cell.alignment = center_alignment

        # 列宽设置
        worksheet.column_dimensions['A'].width = 45
        worksheet.column_dimensions['B'].width = 8
        worksheet.column_dimensions['C'].width = 45
        worksheet.column_dimensions['D'].width = 8
        worksheet.column_dimensions['E'].width = 45
        worksheet.column_dimensions['F'].width = 8
        worksheet.column_dimensions['G'].width = 45
        worksheet.column_dimensions['H'].width = 8
        for col in ['I','J','K','L','M','N','O','P']:
            worksheet.column_dimensions[col].width = 10

        # ========== 外侧框线设置 ==========
        # 1. 1-4列学科统计区域（A-D）
        set_outside_border(worksheet, min_row=1, max_row=sum_row_left, min_col=1, max_col=4)
        # 2. 5-8列QA分类统计区域（E-H）
        set_outside_border(worksheet, min_row=1, max_row=sum_row_right, min_col=5, max_col=8)
        # 3. 总体统计区域（I-P）
        set_outside_border(worksheet, min_row=1, max_row=overall_max_row, min_col=9, max_col=16)

    print(f"统计完成！专业版结果已保存至: {output_path}")

if __name__ == "__main__":
    main()

正在分析 JSON 文件...
JSON文件中共有 703 个键（论文）
调试样例 1: primary='Economics', secondary='Theoretical Economics'
调试样例 2: primary='Economics', secondary='Theoretical Economics'
调试样例 3: primary='Economics', secondary='Theoretical Economics'
调试样例 4: primary='Economics', secondary='Theoretical Economics'
调试样例 5: primary='Statistics', secondary='Other Statistics'

--- 统计结果 ---
总QA数: 6204
成功计数的论文数: 703
各 primary 论文总数: {'Economics': 38, 'Statistics': 82, 'Electrical Engineering and Systems Science': 49, 'Quantitative Biology': 47, 'Quantitative Finance': 28, 'Computer Science': 163, 'Mathematics': 95, 'Physics': 201}
各 primary QA总数: {'Economics': 353, 'Statistics': 754, 'Electrical Engineering and Systems Science': 452, 'Quantitative Biology': 432, 'Quantitative Finance': 251, 'Computer Science': 1495, 'Mathematics': 730, 'Physics': 1737}
选择题: 3984, 简答题: 2220
正在构建左侧学科统计表...
正在构建右侧问题分类统计表...
正在构建总体统计表...
统计完成！专业版结果已保存至: 统计结果.xlsx
